# CineSignal — Train on ml-32m with a Google Colab GPU

Trains all 5 recommenders (popularity, collaborative filtering, content-based,
hybrid, neural) on the **full ml-32m MovieLens dataset** (32,000,204 real
ratings) and saves the fitted artifacts to Google Drive.

**Before you start:** `Runtime > Change runtime type > T4 GPU`.

**Workflow (important):** everything needed to get going -- Drive mount,
the project code, the dataset, dependencies, and the checkpoint path -- is
set up by **ONE cell** (Setup, right below). Free-tier Colab disconnects
sometimes, and a disconnect always wipes `/content/` and resets your Python
session. When that happens: just re-run the **Setup** cell again (nothing
else) -- it's idempotent and reuses everything already cached in Drive
(project zip, dataset, downloaded packages), so the second time is fast --
then re-run the **Train** cell with `--resume`.


## Setup — run this first, and again after any disconnect

Change `DRIVE_PROJECT_DIR` below if you want a different Drive folder.
Everything else in this cell is safe to re-run any number of times: it
checks what's already done (zip cached? dataset cached? packages
installed?) and skips straight past anything that's already there.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, shutil, zipfile, subprocess, sys

DRIVE_PROJECT_DIR = '/content/drive/MyDrive/CineSignal'   # <- change if you want a different Drive folder
os.makedirs(DRIVE_PROJECT_DIR, exist_ok=True)

# checkpoint path -- defined here, right at the top, so it's available to
# every cell below without hunting through the notebook for it.
NEURAL_CKPT_DIR = os.path.join(DRIVE_PROJECT_DIR, 'checkpoints', 'neural_ncf')

PROJECT_DIR = '/content/recsys'

# ---- 1. project code: reuse from Drive if cached, else ask for an upload once ----
zip_in_drive = os.path.join(DRIVE_PROJECT_DIR, 'CineSignal.zip')
local_zip = '/content/CineSignal.zip'

if not os.path.exists(PROJECT_DIR):
    if os.path.exists(zip_in_drive):
        shutil.copy(zip_in_drive, local_zip)
        print('[code] copied project zip from Drive cache')
    elif os.path.exists(local_zip):
        print('[code] using project zip already in this session')
    else:
        print('[code] no cached zip found -- please choose your project zip file:')
        from google.colab import files
        uploaded = files.upload()
        fname = next(iter(uploaded.keys()))
        shutil.move(f'/content/{fname}', local_zip)
        shutil.copy(local_zip, zip_in_drive)
        print('[code] uploaded and cached to Drive -- future sessions will skip this upload')

    with zipfile.ZipFile(local_zip) as zf:
        zf.extractall('/content/')
    print('[code] extracted to', PROJECT_DIR)
else:
    print('[code] already extracted at', PROJECT_DIR)

os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
print('[code] cwd:', os.getcwd())

# ---- 2. dataset: link from Drive cache if present, else download once ----
drive_data_dir = os.path.join(DRIVE_PROJECT_DIR, 'data_raw_ml-32m')
local_data_dir = os.path.join(PROJECT_DIR, 'data', 'raw', 'ml-32m')

if os.path.exists(os.path.join(local_data_dir, 'ratings.csv')):
    print('[data] already present in this session')
elif os.path.exists(os.path.join(drive_data_dir, 'ratings.csv')):
    os.makedirs(os.path.dirname(local_data_dir), exist_ok=True)
    if os.path.islink(local_data_dir) or os.path.exists(local_data_dir):
        shutil.rmtree(local_data_dir, ignore_errors=True)
    os.symlink(drive_data_dir, local_data_dir)
    print('[data] linked from Drive cache (no download needed)')
else:
    print('[data] downloading ml-32m from GroupLens (one-time, ~240MB)...')
    subprocess.run(['python', 'scripts/download_dataset.py'], check=True)
    print('[data] caching a copy to Drive so future sessions skip the download...')
    shutil.copytree(local_data_dir, drive_data_dir)
    print('[data] done')

# ---- 3. dependencies (skip torch -- Colab's GPU runtime already has a CUDA build) ----
subprocess.run("grep -v '^torch' requirements.txt > /tmp/requirements_colab.txt", shell=True, check=True)
subprocess.run(['pip', 'install', '-q', '-r', '/tmp/requirements_colab.txt'], check=True)

import torch
print(f'[env] torch {torch.__version__} | CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print('[env] GPU:', torch.cuda.get_device_name(0))
    print('\n\u2705 Setup complete -- ready to train.')
else:
    print('\n\u26a0\ufe0f No GPU detected. Go to Runtime > Change runtime type > T4 GPU, '
          'Save, then re-run this Setup cell.')

print('\nNeural checkpoints will be written to:', NEURAL_CKPT_DIR)


## Validate the data (quick, optional but recommended once)

In [ ]:
!python -m src.data.validator


## Train

Fits all 5 recommenders on the same train split and evaluates them on the
same held-out test split. Popularity / content-based / collaborative finish
in a few minutes even at 32M rows (sparse matrices, no GPU needed). The
neural (NCF) model is the GPU-bound step -- it prints a line per epoch
(`[NCF] epoch 3/15  train_rmse=...  val_rmse=...`) and checkpoints to
`NEURAL_CKPT_DIR` on Drive:

- `latest.pt` -- overwritten every epoch, and automatically several times
  *within* an epoch too for a dataset this size (so even losing the
  connection minutes into a long epoch only costs a few minutes of work)
- `best.pt` -- only overwritten when validation loss actually improves

Run this for a **fresh** run (no earlier progress to continue):


In [ ]:
!python scripts/train_and_evaluate.py --checkpoint-dir "$NEURAL_CKPT_DIR"


### If Colab disconnects mid-training

1. Reconnect the runtime.
2. Re-run the **Setup** cell above (nothing else) -- it reuses everything
   cached in Drive, so it's quick.
3. Run this cell **instead of** the fresh-run cell above -- `--resume`
   picks up exactly where the neural model left off using
   `NEURAL_CKPT_DIR/latest.pt`, instead of restarting from epoch 1
   (the other 4 models refit quickly regardless, since they aren't
   GPU/epoch-based):


In [ ]:
!python scripts/train_and_evaluate.py --checkpoint-dir "$NEURAL_CKPT_DIR" --resume


## Copy trained artifacts + results back to Drive

The neural checkpoints already live on Drive throughout training, but the
**final** artifacts (popularity/content/collaborative `.joblib` files, the
final `neural_ncf.pt` inference file, and the evaluation results JSON) are
written locally to `/content/recsys/artifacts` -- copy them to Drive so you
can pull them down afterwards to run the Streamlit app locally with
pretrained models instead of retraining.


In [ ]:
drive_artifacts_dir = os.path.join(DRIVE_PROJECT_DIR, 'artifacts_models_ml-32m')
shutil.copytree(os.path.join(PROJECT_DIR, 'artifacts', 'models', 'ml-32m'), drive_artifacts_dir, dirs_exist_ok=True)

drive_docs_dir = os.path.join(DRIVE_PROJECT_DIR, 'docs')
os.makedirs(drive_docs_dir, exist_ok=True)
for fname in ['dataset_summary.json', 'evaluation_results_ml-32m.json']:
    src = os.path.join(PROJECT_DIR, 'docs', fname)
    if os.path.exists(src):
        shutil.copy(src, drive_docs_dir)

print('Copied artifacts to', drive_artifacts_dir)
print('Copied docs to', drive_docs_dir)
print()
print('To use these locally: copy the contents of', drive_artifacts_dir,
      'into your local project at artifacts/models/ml-32m/, and the docs/*.json')
print('files into your local docs/ folder. Then `streamlit run app/streamlit_app.py`')
print('will load the pretrained models instantly instead of retraining.')


## Notes

- **Batch size / epochs**: tune `NEURAL_BATCH_SIZE` and `NEURAL_EPOCHS` in
  `src/config.py` for your GPU. A T4 (free tier) comfortably handles batch
  sizes in the thousands for this small embedding-MLP model.
- **More frequent checkpoints**: if disconnects keep happening very early
  (before even one mid-epoch checkpoint lands), add
  `--save-every-steps 300` (or lower) to either training command above to
  checkpoint more often at the cost of slightly slower training.
- **Keep the browser tab open/active** -- free-tier Colab disconnects on
  prolonged inactivity, separate from the fixed session-length limit.
- **Re-running just the neural model** directly instead of the full
  `train_and_evaluate.py` script (e.g. after tuning hyperparameters):

  ```python
  from src.models.neural import NeuralRecommender

  # fresh run, checkpointing to Drive:
  neural = NeuralRecommender(epochs=15)
  neural.fit(train, val, checkpoint_dir=NEURAL_CKPT_DIR)

  # resuming one that got disconnected:
  neural = NeuralRecommender.resume(
      f"{NEURAL_CKPT_DIR}/latest.pt", train, val, epochs=15,
  )
  ```
